# NF (MEGA) – stellar mass + |pos| + |vel|

Fully self-contained notebook. Runs end-to-end:

1. Load CAMELS-LH groups catalogs for **Astrid, IllustrisTNG, SIMBA, SWIFT-EAGLE**.

2. Build the feature matrix, split 80/20 with seed=42.
3. **Optuna search** (30 trials) over hidden width, depth, spline bins and learning rate; saves `optuna_study.pkl`.
4. Final fit (120 epochs) on the best config, save the checkpoint.
5. Draw 1000 posterior samples per test halo, save predictions.
6. Report metrics, residual plots, mass-binned diagnostics, TARP coverage.

 Each halo's central galaxy contributes its stellar mass (and, where enabled, ugriz+Y magnitudes) as context for a conditional spline flow predicting $\log_{10}\,M_{\rm halo}$.


In [ ]:
import os
import sys
import glob
import time
import json
from dataclasses import dataclass, asdict

import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import joblib
import optuna
import pyro
import pyro.distributions as dist
from pyro.distributions.transforms import conditional_spline
from pyro.distributions import ConditionalTransformedDistribution

plt.rcParams.update({"font.size": 11, "figure.figsize": (9, 5.5)})
np.set_printoptions(precision=4, suppress=True)

# Reproducibility
SPLIT_SEED = 42
np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)
pyro.set_rng_seed(SPLIT_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ---- Paths + constants ----
NB_DIR   = os.path.dirname(os.path.abspath("model.ipynb")) if False else os.getcwd()
REPO_ROOT = os.path.abspath("../..")
OUT_DIR  = NB_DIR                       # write checkpoints/plots alongside the notebook
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)

SIM_DIRS = {
    "Astrid": "data",
    "IllustrisTNG": "IllustrisTNG",
    "SIMBA": "SIMBA",
    "SWIFT-EAGLE": "Swift"
}

# photometry file name prefixes
PHOTO_PREFIX = {
    "Astrid": "Astrid",
    "IllustrisTNG": "IllustrisTNG",
    "SIMBA": "Simba",
    "SWIFT-EAGLE": "Swift-EAGLE"
}

PHOTO_BANDS = [
    ("SLOAN", "SDSS.g"),
    ("SLOAN", "SDSS.r"),
    ("SLOAN", "SDSS.i"),
    ("SLOAN", "SDSS.z"),
    ("UKIRT", "UKIDSS.Y"),
]
PHOTO_DEFAULTS = dict(snap="090", stellar_pop="BC03",
                      kind="luminosity", variant="attenuated")

MASS_CUT = 1.3e8                        # Msun/h lower cut on subhalo stellar mass
MAG_REF  = 30.0                         # AB magnitude zero-point for -2.5 log10(L) - MAG_REF

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"OUT_DIR:   {OUT_DIR}")
print(f"simulations: {list(SIM_DIRS)}  (with_photometry=false)")


In [ ]:
# ============================================================
# LOAD CAMELS-LH GROUPS FILES
# ============================================================
# For each simulation, iterate over the LH realisations and collect
# central galaxies that pass the stellar-mass cut.  We track the halo
# id, galaxy id, LH index and simulation label so posterior predictions
# remain reproducible and traceable.

def _lh_index(path: str) -> int:
    """Recover the CAMELS LH realisation index from a groups file name."""
    base = os.path.basename(path)
    if base == "groups_090.hdf5":
        return 0
    return int(base.split("(")[1].split(")")[0])


sim_all, lh_all, hid_all, gid_all = [], [], [], []
sm_all, rhm_all, sfr_all = [], [], []
log_pos_all, log_vel_all = [], []
log_hm_all = []

total_halos = total_centrals = 0

print("Loading groups catalogs ...")
for sim, sim_dir in SIM_DIRS.items():
    pattern = os.path.join(REPO_ROOT, sim_dir, "groups_090*.hdf5")
    files = sorted(glob.glob(pattern))
    print(f"  [{sim:12s}]  {len(files):2d} realisations")
    for path in files:
        lh = _lh_index(path)
        with h5py.File(path, "r") as f:
            group_first_sub = f["Group/GroupFirstSub"][:]
            group_n_subs    = f["Group/GroupNsubs"][:]
            halo_mass       = f["Group/Group_M_Crit200"][:]    # 1e10 Msun/h
            subhalo_mass_t  = f["Subhalo/SubhaloMassType"][:]
            stellar_mass    = subhalo_mass_t[:, 4] * 1e10      # Msun/h
            sfr             = f["Subhalo/SubhaloSFR"][:] * 1e10
            galaxy_radius_t = f["Subhalo/SubhaloHalfmassRadType"][:, 4]
            galaxy_pos      = f["Subhalo/SubhaloPos"][:]
            galaxy_vel      = f["Subhalo/SubhaloVel"][:]

            n_halos = len(group_first_sub)
            total_halos += n_halos
            for h_idx in range(n_halos):
                first_sub = group_first_sub[h_idx]
                if first_sub == -1:
                    continue
                # A halo contributes one row (its central galaxy) iff its first
                # subhalo passes the stellar-mass cut.
                if stellar_mass[first_sub] <= MASS_CUT:
                    continue
                # |pos| / |vel| of the central relative to the box origin
                # (mirrors shared.data.load_centrals).
                total_centrals += 1
                sim_all.append(sim)
                lh_all.append(lh)
                hid_all.append(int(h_idx))
                gid_all.append(int(first_sub))
                sm_all.append(float(stellar_mass[first_sub]))
                rhm_all.append(float(galaxy_radius_t[first_sub]))
                sfr_all.append(float(sfr[first_sub]))
                log_pos_all.append(float(np.log10(np.linalg.norm(galaxy_pos[first_sub]) + 1e-30)))
                log_vel_all.append(float(np.log10(np.linalg.norm(galaxy_vel[first_sub]) + 1e-30)))
                # Group_M_Crit200 is in 1e10 Msun/h, so multiply by 1e10 to get Msun/h.
                log_hm_all.append(float(np.log10(halo_mass[h_idx] * 1e10)))

sim_arr = np.asarray(sim_all)
lh_arr  = np.asarray(lh_all,  dtype=np.int32)
hid_arr = np.asarray(hid_all, dtype=np.int32)
gid_arr = np.asarray(gid_all, dtype=np.int32)
sm_arr  = np.asarray(sm_all,  dtype=np.float64)
rhm_arr = np.asarray(rhm_all, dtype=np.float64)
sfr_arr = np.asarray(sfr_all, dtype=np.float64)
log_pos_arr = np.asarray(log_pos_all, dtype=np.float64)
log_vel_arr = np.asarray(log_vel_all, dtype=np.float64)
log_hm_arr  = np.asarray(log_hm_all,  dtype=np.float64)

# Derived arrays
log_sm_arr  = np.log10(sm_arr)
log_rhm_arr = np.log10(rhm_arr + 1e-12)

print(f"\nTotal halos scanned: {total_halos:,}")
print(f"Centrals kept (first subhalo above M* cut): {total_centrals:,}")


In [ ]:
# ============================================================
# BUILD FEATURE MATRIX
# ============================================================
# Columns (in order):
#   - log_stellar_mass
#   - log_pos_modulus
#   - log_vel_modulus

feature_columns = ["log_stellar_mass", "log_pos_modulus", "log_vel_modulus"]

X_cols = [log_sm_arr.reshape(-1, 1), log_pos_arr.reshape(-1, 1), log_vel_arr.reshape(-1, 1)]
X = np.concatenate(X_cols, axis=1).astype(np.float64)
y = log_hm_arr.reshape(-1, 1).astype(np.float64)

print(f"X shape: {X.shape}  y shape: {y.shape}")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]  mean={y.mean():.2f}")


In [ ]:
# ============================================================
# NORMALISE + TRAIN/TEST SPLIT (80/20)
# ============================================================
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0); X_std[X_std == 0] = 1.0
y_mean = y.mean(); y_std = y.std()

X_norm = (X - X_mean) / X_std
y_norm = (y - y_mean) / y_std

rng = np.random.default_rng(SPLIT_SEED)
idx = rng.permutation(len(X_norm))
n_train = int(0.8 * len(idx))
train_idx, test_idx = idx[:n_train], idx[n_train:]

X_train = torch.tensor(X_norm[train_idx], dtype=torch.float32, device=device)
y_train = torch.tensor(y_norm[train_idx], dtype=torch.float32, device=device)
X_test  = torch.tensor(X_norm[test_idx],  dtype=torch.float32, device=device)
y_test  = torch.tensor(y_norm[test_idx],  dtype=torch.float32, device=device)

print(f"train={len(train_idx):,}  test={len(test_idx):,}")
print(f"X_mean: {X_mean}")
print(f"X_std:  {X_std}")
print(f"y_mean: {y_mean:.4f}  y_std: {y_std:.4f}")


In [ ]:
# ============================================================
# OPTUNA HYPERPARAMETER SEARCH
# ============================================================
# Search the spline flow's depth (n_transforms), bin count, hidden
# layer width, and learning rate.  Each trial trains a short flow on
# the train split and reports the held-out NLL on the test split.
# Pruning kills bad trials early.

input_dim   = 1
context_dim = X.shape[1]
bound       = 5.0


def _build_flow(cfg, device):
    cts = []
    for _ in range(cfg["n_transforms"]):
        t = conditional_spline(
            input_dim=input_dim, context_dim=context_dim,
            hidden_dims=[cfg["hidden"], cfg["hidden"]],
            count_bins=cfg["count_bins"], bound=bound,
        )
        cts.append(t)
    base = dist.Normal(torch.zeros(input_dim, device=device),
                       torch.ones(input_dim, device=device))
    flow = ConditionalTransformedDistribution(base, cts)
    mods = nn.ModuleList(cts).to(device)
    return mods, flow, cts


def objective(trial):
    cfg = dict(
        hidden       = trial.suggest_categorical("hidden",       [64, 128, 256]),
        n_transforms = trial.suggest_int       ("n_transforms",  4, 12),
        count_bins   = trial.suggest_categorical("count_bins",   [8, 16, 24]),
    )
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    mods, flow, _ = _build_flow(cfg, device)
    opt = optim.Adam(mods.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=512, shuffle=True)
    n_search_epochs = 25
    best = float("inf")
    for epoch in range(n_search_epochs):
        mods.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = -flow.condition(xb).log_prob(yb).mean()
            loss.backward(); opt.step()
        mods.eval()
        with torch.no_grad():
            vloss = -flow.condition(X_test).log_prob(y_test).mean().item()
        best = min(best, vloss)
        trial.report(vloss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return best


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=0),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=30, show_progress_bar=True)
joblib.dump(study, os.path.join(OUT_DIR, "optuna_study.pkl"))
best = study.best_params
print(f"\nbest params: {best}")
print(f"best value (NLL): {study.best_value:.4f}")
print(f"saved -> {os.path.join(OUT_DIR, 'optuna_study.pkl')}")


In [ ]:
# ============================================================
# BUILD NF FROM BEST PARAMS
# ============================================================
hidden_dims  = [best["hidden"], best["hidden"]]
n_transforms = best["n_transforms"]
count_bins   = best["count_bins"]
learning_rate = best["lr"]

modules, flow_dist, cond_transforms = _build_flow(
    dict(hidden=best["hidden"], n_transforms=n_transforms, count_bins=count_bins),
    device,
)
n_params = sum(p.numel() for p in modules.parameters())
print(f"context_dim={context_dim}  hidden={hidden_dims}  transforms={n_transforms}"
      f"  bins={count_bins}  lr={learning_rate:.2e}  params={n_params:,}")


In [ ]:
# ============================================================
# FINAL TRAINING (with the Optuna-best params)
# ============================================================
batch_size    = 512
n_epochs      = 120
# learning_rate already set from best["lr"] in the previous cell

dataset = TensorDataset(X_train, y_train)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
opt = optim.Adam(modules.parameters(), lr=learning_rate)

train_losses, test_losses = [], []
t0 = time.time()
for epoch in range(n_epochs):
    modules.train()
    running, nb = 0.0, 0
    for xb, yb in loader:
        opt.zero_grad()
        loss = -flow_dist.condition(xb).log_prob(yb).mean()
        loss.backward(); opt.step()
        running += loss.item(); nb += 1
    train_losses.append(running / max(1, nb))
    modules.eval()
    with torch.no_grad():
        tl = -flow_dist.condition(X_test).log_prob(y_test).mean().item()
    test_losses.append(tl)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"epoch {epoch+1:3d}/{n_epochs} | train {train_losses[-1]:+.4f} | test {tl:+.4f}")
print(f"\nTraining done in {time.time() - t0:.1f}s")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(train_losses, label="train")
ax.plot(test_losses, label="test")
ax.set_xlabel("epoch"); ax.set_ylabel("NLL")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"NF (MEGA) – stellar mass + |pos| + |vel| — training curves")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "training_curves.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# SAVE CHECKPOINT
# ============================================================
model_path = os.path.join(OUT_DIR, "best_model.pt")
torch.save({
    "model_state_dict": modules.state_dict(),
    "X_mean": X_mean, "X_std": X_std,
    "y_mean": y_mean, "y_std": y_std,
    "model_params": dict(
        input_dim=input_dim, context_dim=context_dim,
        hidden_dims=hidden_dims, n_transforms=n_transforms,
        count_bins=count_bins, bound=bound,
    ),
    "cond_transforms": cond_transforms,
    "feature_names": feature_columns + ([]),
    "model_name": "NF (MEGA) \u2013 stellar mass + |pos| + |vel|",
}, model_path)
print(f"saved -> {model_path}")


In [ ]:
# ============================================================
# SAMPLE POSTERIOR FOR TEST SET  (1000 samples/halo)
# ============================================================
# We tile the context across the sample dim so the spline's per-row
# parameters share the batch dimension with the base-dist draws.

n_posterior = 1000
batch_size_infer = 64

X_test_t = X_test
samples = np.zeros((len(X_test_t), n_posterior), dtype=np.float32)
modules.eval()
with torch.no_grad():
    for start in range(0, len(X_test_t), batch_size_infer):
        end = min(start + batch_size_infer, len(X_test_t))
        ctx = X_test_t[start:end]
        B, F = ctx.shape
        k = n_posterior * B
        ctx_rep = ctx.repeat_interleave(n_posterior, dim=0)
        cond = flow_dist.condition(ctx_rep)
        s = cond.sample([k]).cpu().numpy()
        s = s.squeeze(-1).reshape(B, n_posterior)
        samples[start:end] = s * y_std + y_mean

pred_mean = samples.mean(axis=1)
pred_std  = samples.std(axis=1)

y_true   = y[test_idx].flatten()
print(f"posterior shape: {samples.shape}")
print(f"pred range: [{pred_mean.min():.2f}, {pred_mean.max():.2f}]"
      f"  truth range: [{y_true.min():.2f}, {y_true.max():.2f}]")


In [ ]:
# ============================================================
# SAVE POSTERIOR PREDICTIONS
# ============================================================
pred_path = os.path.join(OUT_DIR, "test_predictions.npz")
np.savez_compressed(
    pred_path,
    simulation=sim_arr[test_idx],
    halo_id=hid_arr[test_idx],
    galaxy_id=gid_arr[test_idx],
    lh_index=lh_arr[test_idx],
    true_log_halo_mass=y_true.astype(np.float32),
    posterior_log_halo_mass=samples.astype(np.float32),
    posterior_mean=pred_mean.astype(np.float32),
    posterior_std=pred_std.astype(np.float32),
    feature_names=np.array(feature_columns + ([])),
    feature_values=X[test_idx].astype(np.float32),
    n_posterior=int(n_posterior),
    model_name="NF (MEGA) \u2013 stellar mass + |pos| + |vel|",
)
print(f"saved -> {pred_path}")


In [ ]:
# ============================================================
# METRICS (RMSE, chi2, PCC, eps, R^2)
# ============================================================
def compute_metrics(y, mu, sigma):
    y, mu, sigma = map(np.asarray, (y, mu, sigma))
    residual = mu - y
    rmse  = float(np.sqrt(np.mean(residual ** 2)))
    chi2  = float(np.mean((residual / np.maximum(sigma, 1e-6)) ** 2))
    if y.std() == 0 or mu.std() == 0:
        pcc = float("nan")
    else:
        pcc = float(np.corrcoef(y, mu)[0, 1])
    eps   = float(np.mean(np.abs(10 ** mu - 10 ** y) / (10 ** y)) * 100.0)
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return dict(rmse=rmse, chi2=chi2, pcc=pcc, eps=eps, r2=r2)

overall = compute_metrics(y_true, pred_mean, pred_std)
print(f"Overall (N={len(y_true)}):")
print(f"  RMSE = {overall['rmse']:.4f}")
print(f"  chi2 = {overall['chi2']:.2f}")
print(f"  PCC  = {overall['pcc']:.4f}")
print(f"  eps  = {overall['eps']:.2f}%")
print(f"  R^2  = {overall['r2']:.4f}")

print("\n--- Per simulation ---")
for s in sorted(set(sim_arr[test_idx].tolist())):
    sel = sim_arr[test_idx] == s
    if sel.sum() == 0:
        continue
    m = compute_metrics(y_true[sel], pred_mean[sel], pred_std[sel])
    print(f"  {s:14s}  N={sel.sum():5d}  RMSE={m['rmse']:.4f}  chi2={m['chi2']:.2f}"
          f"  PCC={m['pcc']:.3f}  eps={m['eps']:.1f}%  R2={m['r2']:.3f}")



In [ ]:
# ============================================================
# SAVE metrics.json + hyperparameters.txt
# ============================================================
metrics_payload = {"model": "NF (MEGA) \u2013 stellar mass + |pos| + |vel|", "overall": overall, "per_simulation": {}}
for s in sorted(set(sim_arr[test_idx].tolist())):
    sel = sim_arr[test_idx] == s
    if sel.sum() == 0:
        continue
    metrics_payload["per_simulation"][s] = compute_metrics(
        y_true[sel], pred_mean[sel], pred_std[sel]
    )
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as fh:
    json.dump(metrics_payload, fh, indent=2)
print(f"saved -> {os.path.join(OUT_DIR, 'metrics.json')}")

hparams = {
    "model_name":      "NF (MEGA) \u2013 stellar mass + |pos| + |vel|",
    "feature_columns": feature_columns + ([]),
    "context_dim":     int(context_dim),
    "input_dim":       int(input_dim),
    "hidden_dims":     hidden_dims,
    "n_transforms":    int(n_transforms),
    "count_bins":      int(count_bins),
    "bound":           float(bound),
    "learning_rate":   float(learning_rate),
    "batch_size":      int(batch_size),
    "n_epochs":        int(n_epochs),
    "split_seed":      int(SPLIT_SEED),
    "n_train":         int(len(train_idx)),
    "n_test":          int(len(test_idx)),
    "final_train_loss": float(train_losses[-1]),
    "final_test_loss":  float(test_losses[-1]),
}
with open(os.path.join(OUT_DIR, "hyperparameters.txt"), "w") as fh:
    fh.write(f"# Hyperparameters for {hparams['model_name']}\n")
    for k, v in hparams.items():
        fh.write(f"{k}: {v}\n")
print(f"saved -> {os.path.join(OUT_DIR, 'hyperparameters.txt')}")


In [ ]:
# ============================================================
# RESIDUAL PANEL (combined)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_true, pred_mean - y_true, s=6, alpha=0.3)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[0].set_ylabel("Predicted − True")
axes[0].set_title(f"NF (MEGA) – stellar mass + |pos| + |vel| — residuals (N={len(y_true)})")
axes[0].grid(alpha=0.3)

axes[1].hist(pred_mean - y_true, bins=60, color="steelblue", alpha=0.8, edgecolor="k")
axes[1].axvline(0, color="k", lw=1)
mu_r = float(np.mean(pred_mean - y_true))
sd_r = float(np.std(pred_mean - y_true))
axes[1].set_xlabel("Predicted − True")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Residual dist  (μ={mu_r:+.3f}, σ={sd_r:.3f})")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "residuals_combined.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# PER-SIMULATION RESIDUAL GRID
# ============================================================
sims = sorted(set(sim_arr[test_idx].tolist()))
n = len(sims)
cols = min(n, 2); rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.2 * rows),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()
for ax, s in zip(axes, sims):
    sel = sim_arr[test_idx] == s
    if sel.sum() == 0:
        ax.set_visible(False); continue
    ax.scatter(y_true[sel], pred_mean[sel] - y_true[sel], s=6, alpha=0.35)
    ax.axhline(0, color="k", lw=1)
    m = compute_metrics(y_true[sel], pred_mean[sel], pred_std[sel])
    ax.set_title(f"{s}  N={sel.sum():,}  RMSE={m['rmse']:.3f}  chi²={m['chi2']:.2f}")
    ax.set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
    ax.set_ylabel("Predicted − True")
    ax.grid(alpha=0.3)
for ax in axes[len(sims):]:
    ax.set_visible(False)
fig.suptitle(f"NF (MEGA) – stellar mass + |pos| + |vel| — per simulation", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "residuals_per_simulation.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# METRICS IN MASS BINS
# ============================================================
def metrics_in_bins(y, mu, sigma, n_bins=10):
    edges   = np.linspace(y.min(), y.max(), n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    rmse_b, chi_b = [], []
    for i in range(n_bins):
        sel = (y >= edges[i]) & (y < edges[i + 1])
        if sel.sum() < 2:
            rmse_b.append(np.nan); chi_b.append(np.nan); continue
        rmse_b.append(np.sqrt(np.mean((mu[sel] - y[sel]) ** 2)))
        chi_b.append(np.mean(((mu[sel] - y[sel]) / np.maximum(sigma[sel], 1e-6)) ** 2))
    return centers, np.array(rmse_b), np.array(chi_b)

centers, rmse_b, chi_b = metrics_in_bins(y_true, pred_mean, pred_std)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(centers, rmse_b, "o-", color="steelblue")
axes[0].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[0].set_ylabel("RMSE"); axes[0].grid(alpha=0.3)
axes[0].set_title("RMSE in halo-mass bins")

axes[1].plot(centers, chi_b, "o-", color="darkorange")
axes[1].axhline(1.0, color="k", ls="--", lw=1)
axes[1].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[1].set_ylabel(r"$\chi^2$"); axes[1].grid(alpha=0.3)
axes[1].set_title(r"$\chi^2$ in halo-mass bins")

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "metrics_vs_mass.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# TARP COVERAGE TEST
# ============================================================
try:
    import tarp
    # tarp expects samples shaped (n_samples, n_sims, n_params)
    tarp_samples = samples.T[:, :, np.newaxis]           # (n_post, N_test, 1)
    theta        = y_true[:, np.newaxis]                 # (N_test, 1)
    ecp, alpha = tarp.get_tarp_coverage(
        tarp_samples, theta,
        references="random", metric="euclidean",
        num_bootstrap=100, bootstrap=True, seed=0,
    )
    # One-sided KS test against the diagonal
    if ecp.ndim == 2:
        ecp_mean = ecp.mean(axis=0)
    else:
        ecp_mean = ecp
    d = np.max(np.abs(ecp_mean - alpha))
    ks_p = float(np.exp(-2 * len(alpha) * d ** 2))

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot(alpha, alpha, "k--", lw=1, label="perfect")
    if ecp.ndim == 2:
        lo, hi = np.quantile(ecp, [0.16, 0.84], axis=0)
        ax.fill_between(alpha, lo, hi, color="C0", alpha=0.25, label="68%")
        ax.plot(alpha, ecp_mean, color="C0", lw=2)
    else:
        ax.plot(alpha, ecp, color="C0", lw=2)
    ax.set_xlabel("Credibility α")
    ax.set_ylabel("Expected coverage prob.")
    ax.text(0.04, 0.96, f"KS p = {ks_p:.3g}", transform=ax.transAxes,
            va="top", bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    ax.set_title(f"NF (MEGA) – stellar mass + |pos| + |vel| — TARP coverage")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "plots", "tarp_coverage.png"), dpi=150)
    plt.show()
except Exception as e:
    print(f"TARP failed: {e}")


## Summary — NF (MEGA) – stellar mass + |pos| + |vel|

Everything above runs end-to-end from raw CAMELS-LH HDF5 files.
Artifacts are written alongside the notebook:

- `best_model.pt` — conditional flow weights + normalisation + metadata
- `test_predictions.npz` — 1000 posterior samples per test halo, plus `simulation`, `halo_id`, `galaxy_id`, `lh_index`, `true_log_halo_mass`
- `plots/training_curves.png`, `plots/residuals_combined.png`, `plots/metrics_vs_mass.png`, `plots/tarp_coverage.png`, `plots/residuals_per_simulation.png`
